# Notebook 09: ML Monitoring and Drift

## Why This Matters

A model deployed to production is not done - it is the beginning. Real-world data distributions shift constantly: user behavior changes seasonally, new product categories emerge, fraud patterns evolve, and society changes. Without monitoring, a model degrades silently. At Stripe, undiscovered model degradation in fraud detection could cost millions per day before anyone noticed. ML monitoring is one of the most underinvested areas in applied ML and a key differentiator between junior and senior engineers.

In [ ]:
%pip install -q scipy numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
np.random.seed(42)
print("Ready.")

## 1. Types of Distribution Shift

**Data drift (covariate shift)**: P(X) changes, but P(Y|X) stays the same. Example: model trained on desktop transactions, mobile usage surges with different feature distributions.

**Concept drift**: P(Y|X) changes - same features now predict different outcomes. Example: user intent behind 'coronavirus' search changed from curiosity (2019) to urgent health need (2020).

**Label shift**: P(Y) changes. Example: fraud rate drops from 1% to 0.1% after a crackdown.

**Feedback loops**: Your model's predictions change user behavior, which changes the data distribution.

**Practical impact**: Data drift degrades recall. Concept drift degrades precision. Monitor both.

In [ ]:
def generate_stream(n_days=60, n_per_day=500, drift_day=30):
    stream = []
    for day in range(n_days):
        if day < drift_day:
            X, y = make_classification(n_samples=n_per_day, n_features=5, n_informative=2, weights=[0.95, 0.05], random_state=day)
        else:
            X, y = make_classification(n_samples=n_per_day, n_features=5, n_informative=2, weights=[0.92, 0.08], random_state=day+1000, flip_y=0.1)
            X[:, 0] += 1.5  # feature distribution shift
        stream.append((day, X, y))
    return stream

stream = generate_stream()
train_Xs = [X for day, X, y in stream[:10]]; train_ys = [y for day, X, y in stream[:10]]
model = LogisticRegression(max_iter=500).fit(np.vstack(train_Xs), np.hstack(train_ys))

metrics = []
for day, X, y in stream:
    preds = model.predict_proba(X)[:,1]
    auc = roc_auc_score(y, preds) if y.sum() > 0 and y.sum() < len(y) else np.nan
    metrics.append({"day": day, "auc": auc, "pos_rate": y.mean(), "mean_f0": X[:,0].mean()})
df_m = pd.DataFrame(metrics)

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
for ax, col, title in [(axes[0],"auc","Model AUC"), (axes[1],"pos_rate","Positive Rate"), (axes[2],"mean_f0","Feature 0 Mean")]:
    ax.plot(df_m.day, df_m[col]); ax.axvline(30, color="red", linestyle="--", label="Drift point")
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig("/tmp/drift.png", dpi=80); plt.show()
print(f"Pre-drift AUC: {df_m[df_m.day<30].auc.mean():.4f}")
print(f"Post-drift AUC: {df_m[df_m.day>=30].auc.mean():.4f}")

## 2. Statistical Tests for Drift

**KS test**: Non-parametric test comparing two empirical distributions. Small p-value = distributions are different.

**PSI (Population Stability Index)**: Industry standard in banking.
`PSI = sum (actual% - expected%) * ln(actual% / expected%)`
- PSI < 0.1: no significant change
- 0.1 <= PSI < 0.25: moderate change, monitor
- PSI >= 0.25: significant change, investigate and potentially retrain

In [ ]:
def compute_ks(reference, current):
    stat, p = stats.ks_2samp(reference, current)
    return {"ks_stat": stat, "p_value": p, "drifted": p < 0.05}

def compute_psi(reference, current, bins=10):
    edges = np.percentile(reference, np.linspace(0, 100, bins+1))
    edges[0] -= 1e-6; edges[-1] += 1e-6
    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)
    ref_pct = ref_counts / len(reference) + 1e-6
    cur_pct = cur_counts / len(current) + 1e-6
    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    status = "No drift" if psi < 0.1 else ("Moderate" if psi < 0.25 else "SIGNIFICANT - investigate")
    return {"psi": psi, "status": status}

reference_X = stream[5][1]
print("=== Drift Detection Results ===\n")
for check_day in [20, 30, 40, 50]:
    current_X = stream[check_day][1]
    print(f"Day {check_day}:")
    for feat_idx in range(5):
        ks = compute_ks(reference_X[:,feat_idx], current_X[:,feat_idx])
        psi = compute_psi(reference_X[:,feat_idx], current_X[:,feat_idx])
        flag = " <- DRIFT DETECTED" if (ks["drifted"] or psi["psi"] >= 0.1) else ""
        print(f"  Feature {feat_idx}: KS p={ks["p_value"]:.4f}, PSI={psi["psi"]:.4f} [{psi["status"]}]{flag}")
    print()

## 3. Monitoring Pipeline Design

A production monitoring pipeline:
1. **Feature logging**: log input features + model predictions for every inference
2. **Batch drift jobs**: hourly/daily statistical tests against training distribution baseline
3. **Alerting**: page on-call when PSI > 0.25 or AUC drops > 5%
4. **Shadow deployment**: run new model alongside old, compare outputs
5. **Retraining triggers**: drift PSI > 0.25, or scheduled weekly retrain on rolling window

**Label delay problem**: Ground truth not available for hours/days. Solutions:
- Monitor feature drift as proxy (available immediately)
- Use proxy labels (chargebacks instead of confirmed fraud)
- Monitor prediction distribution shift

In [ ]:
class DriftMonitor:
    def __init__(self, reference_data, feature_names, psi_threshold=0.25, ks_alpha=0.05):
        self.reference = reference_data
        self.feature_names = feature_names
        self.psi_threshold = psi_threshold
        self.alert_log = []
    
    def check_batch(self, current_data, batch_id):
        alerts = []
        for i, fname in enumerate(self.feature_names):
            ks = compute_ks(self.reference[:,i], current_data[:,i])
            psi = compute_psi(self.reference[:,i], current_data[:,i])
            if ks["drifted"] or psi["psi"] >= 0.1:
                alerts.append({"batch_id": batch_id, "feature": fname,
                                "psi": psi["psi"], "severity": "HIGH" if psi["psi"] >= 0.25 else "MEDIUM"})
        self.alert_log.extend(alerts)
        return alerts
    
    def generate_report(self):
        if not self.alert_log:
            print("No drift detected - system healthy."); return
        df = pd.DataFrame(self.alert_log)
        print(f"Drift Report: {len(df)} alerts, {df.feature.nunique()} features flagged")
        print(f"HIGH severity: {(df.severity=="HIGH").sum()}")
        print("\nTop drifting features:")
        print(df.groupby("feature")["psi"].max().sort_values(ascending=False).to_string())

monitor = DriftMonitor(stream[0][1], [f"f{i}" for i in range(5)])
for day, X, y in stream[10:]:
    monitor.check_batch(X, f"day_{day}")
monitor.generate_report()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Data drift | P(X) changes; same mapping but inputs are out-of-distribution |
| Concept drift | P(Y|X) changes; same input but correct output changed |
| KS test | Non-parametric distribution comparison; small p-value = drift |
| PSI | Industry standard; PSI < 0.1 stable, 0.1-0.25 moderate, >0.25 significant |
| Label delay | Ground truth arrives late; monitor features as proxy |
| Retraining trigger | PSI > 0.25 or scheduled rolling window retrain |

### Common Pitfalls
- Not logging input features at inference time
- Monitoring only model accuracy (labels arrive late; feature drift is the early warning)
- Retraining on all historical data (ancient data hurts if distribution has shifted)
- Not having a rollback plan when a retrained model performs worse
